# Patch Token Clustering

Extract DINO patch tokens from an image, build a pairwise cosine similarity matrix,
and compare clustering algorithms by overlaying assignments on the original image.

In [ ]:
%pip install -q scikit-learn

In [ ]:
# Logging must be configured before torch is imported.
import logging
import os

logging.basicConfig(
    level=logging.INFO,
    format="%(levelname)s %(name)s: %(message)s",
    force=True,
)
log = logging.getLogger("patch_clustering")

from pathlib import Path
from urllib.request import urlretrieve

import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import torch.nn.functional as F
from PIL import Image
from sklearn.cluster import DBSCAN, AgglomerativeClustering, KMeans, SpectralClustering
from sklearn.preprocessing import normalize as sk_normalize

from dinoisawesome import DinoEncoder
from dotenv import load_dotenv

In [ ]:
# ── Parameters ────────────────────────────────────────────────────────────────
# Set IMAGE_PATH to a local file path string, or leave None to download the example.
load_dotenv()
IMAGE_PATH: str | None = 'DinoisAwesome/data/tiger.jpeg'
SAMPLE_URL = "https://upload.wikimedia.org/wikipedia/commons/thumb/3/3a/Cat03.jpg/1200px-Cat03.jpg"

# Model settings
DINO_VERSION = "v3"  # "v2" or "v3"
DINO_SIZE = "large"  # "small" | "base" | "large" | "giant"
IMG_SIZE = 512  # square input side (must be divisible by patch_size: 14 for v2, 16 for v3)

# Local weights directory: set DINO_WEIGHTS_DIR in the environment to load .pth files
# from disk instead of downloading from torch hub.
# Example: export DINO_WEIGHTS_DIR=/path/to/dino_weights
DINO_WEIGHTS_DIR: str | None = os.environ.get("DINO_WEIGHTS_DIR")

# Clustering parameters
N_CLUSTERS = 6  # k for KMeans, Agglomerative, Spectral
DBSCAN_EPS = 0.40  # cosine distance threshold; tune up/down if clusters look wrong

In [ ]:
# ── Load model ─────────────────────────────────────────────────────────────────
encoder = DinoEncoder(
    version=DINO_VERSION,
    size=DINO_SIZE,
    img_size=IMG_SIZE,
    weights_dir=DINO_WEIGHTS_DIR,
)
H_GRID, W_GRID = encoder.grid_h, encoder.grid_w
N_PATCHES = H_GRID * W_GRID
log.info(
    "DINOv%s-%s | patch_size=%d | grid=%dx%d (%d patches)",
    DINO_VERSION[1],
    DINO_SIZE,
    encoder.patch_size,
    H_GRID,
    W_GRID,
    N_PATCHES,
)

In [ ]:
# ── Load image ─────────────────────────────────────────────────────────────────
IMAGE_PATH: str | None = 'DinoisAwesome/data/tiger.jpeg'
if IMAGE_PATH is not None:
    img = Image.open(IMAGE_PATH).convert("RGB")
    log.info("Loaded image from %s", IMAGE_PATH)
else:
    _cache = Path("/tmp/dino_sample.jpg")
    if not _cache.exists():
        log.info("Downloading sample image...")
        urlretrieve(SAMPLE_URL, _cache)
    img = Image.open(_cache).convert("RGB")
    log.info("Sample image: %dx%d px", img.width, img.height)

fig, ax = plt.subplots(figsize=(5, 5))
ax.imshow(img)
ax.set_title("Input image")
ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# ── Extract patch tokens ───────────────────────────────────────────────────────
output = encoder(img, debias=True)
# output.patches shape: (1, H_GRID, W_GRID, D) for single-layer extraction
patches_grid = output.patches[0]  # (H, W, D)
H, W, D = patches_grid.shape
tokens = patches_grid.reshape(N_PATCHES, D)  # (N, D)
log.info("Tokens: shape=%s  dtype=%s  device=%s", tokens.shape, tokens.dtype, tokens.device)

In [ ]:
# ── Cross-similarity and distance matrices ─────────────────────────────────────
tokens_norm = F.normalize(tokens.float(), dim=-1)  # unit vectors
similarity = (tokens_norm @ tokens_norm.T).cpu().numpy()  # (N, N)  in [-1, 1]
distance = np.clip(1.0 - similarity, 0.0, 2.0)  # cosine distance in [0, 2]
affinity = (similarity + 1.0) / 2.0  # non-negative, for SpectralClustering

sim_offdiag = similarity.copy()
np.fill_diagonal(sim_offdiag, np.nan)
log.info(
    "Off-diagonal similarity: min=%.3f  max=%.3f  mean=%.3f",
    float(np.nanmin(sim_offdiag)),
    float(np.nanmax(sim_offdiag)),
    float(np.nanmean(sim_offdiag)),
)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

im0 = axes[0].imshow(similarity, cmap="RdYlGn", vmin=-1, vmax=1, aspect="auto")
axes[0].set_title(f"Cosine similarity  ({N_PATCHES}\u00d7{N_PATCHES} patches)")
axes[0].set_xlabel("patch index")
axes[0].set_ylabel("patch index")
plt.colorbar(im0, ax=axes[0], shrink=0.85)

im1 = axes[1].imshow(distance, cmap="viridis", vmin=0, vmax=2, aspect="auto")
axes[1].set_title(f"Cosine distance  ({N_PATCHES}\u00d7{N_PATCHES} patches)")
axes[1].set_xlabel("patch index")
plt.colorbar(im1, ax=axes[1], shrink=0.85)

plt.tight_layout()
plt.show()

In [ ]:
# ── Define clustering algorithms ───────────────────────────────────────────────
# Each tuple: (display_name, fitted_estimator, input_matrix)
tokens_l2 = sk_normalize(tokens.cpu().float().numpy())  # L2-normed features for KMeans
distance
algo_configs: list[tuple[str, object, np.ndarray]] = [
    (
        f"DBSCAN\neps={DBSCAN_EPS}",
        DBSCAN(eps=DBSCAN_EPS, min_samples=3, metric="precomputed"),
        distance,
    ),
    (
        "Agglomerative\naverage linkage",
        AgglomerativeClustering(n_clusters=N_CLUSTERS, metric="precomputed", linkage="average"),
        distance,
    ),
    (
        "Agglomerative\ncomplete linkage",
        AgglomerativeClustering(n_clusters=N_CLUSTERS, metric="precomputed", linkage="complete"),
        distance,
    ),
    (
        "Agglomerative\nsingle linkage",
        AgglomerativeClustering(n_clusters=N_CLUSTERS, metric="precomputed", linkage="single"),
        distance,
    ),
    (
        f"KMeans  k={N_CLUSTERS}",
        KMeans(n_clusters=N_CLUSTERS, n_init=10, random_state=42),
        tokens_l2,
    ),
    (
        f"Spectral  k={N_CLUSTERS}",
        SpectralClustering(
            n_clusters=N_CLUSTERS, affinity="precomputed", random_state=42, n_init=10
        ),
        affinity,
    ),
]

# HDBSCAN — available in scikit-learn >= 1.3
try:
    from sklearn.cluster import HDBSCAN  # noqa: PLC0415

    algo_configs.append(
        (
            "HDBSCAN\nmin_cluster_size=5",
            HDBSCAN(min_cluster_size=5, metric="precomputed"),
            distance,
        )
    )
    log.info("HDBSCAN available — added to experiment")
except ImportError:
    log.warning("HDBSCAN not available (requires scikit-learn >= 1.3)")

# ── Run all algorithms ─────────────────────────────────────────────────────────
cluster_results: list[tuple[str, np.ndarray]] = []
for name, algo, X in algo_configs:
    labels = algo.fit_predict(X)
    n_found = len(set(labels) - {-1})
    n_noise = int((labels == -1).sum())
    log.info("%-44s  %d clusters  %d noise", name.replace("\n", " "), n_found, n_noise)
    cluster_results.append((name, labels))

In [ ]:
# ── Visualise cluster assignments ──────────────────────────────────────────────
OVERLAY_ALPHA = 0.60
CMAP = plt.get_cmap("tab20")  # 20 visually distinct colours

# Display image resized to the encoder's input resolution so patch boundaries align.
display_img = np.array(img.resize((IMG_SIZE, IMG_SIZE), Image.BICUBIC))


def _labels_to_rgba(labels: np.ndarray) -> np.ndarray:
    """(N,) flat labels → (H_GRID, W_GRID, 4) RGBA float32 overlay at patch resolution."""
    labels_2d = labels.reshape(H_GRID, W_GRID)
    unique_clusters = sorted(c for c in set(labels_2d.flat) if c != -1)
    rgba = np.zeros((H_GRID, W_GRID, 4), dtype=np.float32)
    for i, cid in enumerate(unique_clusters):
        color = np.array(CMAP(i % 20), dtype=np.float32)
        color[3] = OVERLAY_ALPHA
        rgba[labels_2d == cid] = color
    if -1 in labels_2d:
        rgba[labels_2d == -1] = [0.05, 0.05, 0.05, OVERLAY_ALPHA]
    return rgba


def _draw_result(ax: plt.Axes, labels: np.ndarray, title: str) -> None:
    labels_2d = labels.reshape(H_GRID, W_GRID)
    unique_clusters = sorted(c for c in set(labels_2d.flat) if c != -1)
    n_clusters = len(unique_clusters)
    n_noise = int((labels == -1).sum())

    # Scale patch-grid overlay to image resolution with nearest-neighbour to keep hard edges.
    overlay_small = _labels_to_rgba(labels)
    overlay_pil = Image.fromarray((overlay_small * 255).astype(np.uint8), mode="RGBA")
    overlay_arr = np.array(overlay_pil.resize((IMG_SIZE, IMG_SIZE), Image.NEAREST)) / 255.0

    ax.imshow(display_img)
    ax.imshow(overlay_arr, interpolation="nearest")

    legend_handles = [
        mpatches.Patch(color=CMAP(i % 20), label=f"cluster {cid}")
        for i, cid in enumerate(unique_clusters)
    ]
    if n_noise:
        legend_handles.append(
            mpatches.Patch(facecolor=(0.05, 0.05, 0.05), label=f"noise ({n_noise})")
        )
    ax.legend(handles=legend_handles, loc="lower right", fontsize=6, framealpha=0.85)

    suffix = f", {n_noise} noise" if n_noise else ""
    ax.set_title(f"{title}\n{n_clusters} clusters{suffix}", fontsize=9)
    ax.axis("off")


# ── Layout: header row + algorithm rows ───────────────────────────────────────
NCOLS = 3
n_algos = len(cluster_results)
n_algo_rows = (n_algos + NCOLS - 1) // NCOLS
nrows = 1 + n_algo_rows  # header row + algorithm rows

fig, axes = plt.subplots(nrows, NCOLS, figsize=(NCOLS * 5, nrows * 5))
axes = axes.flatten()

# Header row: original | similarity | distance
axes[0].imshow(display_img)
axes[0].set_title("Original image", fontsize=11)
axes[0].axis("off")

im_sim = axes[1].imshow(similarity, cmap="RdYlGn", vmin=-1, vmax=1, aspect="auto")
axes[1].set_title("Cosine similarity matrix", fontsize=11)
axes[1].set_xlabel("patch index")
axes[1].set_ylabel("patch index")
plt.colorbar(im_sim, ax=axes[1], shrink=0.80)

im_dist = axes[2].imshow(distance, cmap="viridis", vmin=0, vmax=2, aspect="auto")
axes[2].set_title("Cosine distance matrix", fontsize=11)
axes[2].set_xlabel("patch index")
plt.colorbar(im_dist, ax=axes[2], shrink=0.80)

# Algorithm subplots
for idx, (name, labels) in enumerate(cluster_results):
    _draw_result(axes[NCOLS + idx], labels, name)

# Hide unused cells
for i in range(NCOLS + n_algos, len(axes)):
    axes[i].axis("off")

plt.suptitle(
    f"DINO{DINO_VERSION}-{DINO_SIZE}  |  {N_PATCHES} patches  |  {IMG_SIZE}\u00d7{IMG_SIZE}px",
    fontsize=13,
    y=1.01,
)
plt.tight_layout()
plt.show()

In [ ]:
from DinoisAwesome.dinoisawesome.foreground_head import ForegroundHead
fg_head = ForegroundHead(encoder, n_components=3)
fg_result = fg_head.segment(img, threshold=-0.2, debias=True)
plt.imshow(fg_result['foreground_mask'])

In [ ]:
# ── Extract patch tokens ───────────────────────────────────────────────────────
output = encoder(img, debias=True)
# output.patches shape: (1, H_GRID, W_GRID, D) for single-layer extraction
patches_grid = output.patches[0][~fg_result['foreground_mask_feature']]  # (H, W, D)
N, D = patches_grid.shape
tokens = patches_grid.reshape(N, D)  # (N, D)
log.info("Tokens: shape=%s  dtype=%s  device=%s", tokens.shape, tokens.dtype, tokens.device)

In [ ]:
# ── Cross-similarity and distance matrices ─────────────────────────────────────
tokens_norm = F.normalize(tokens.float(), dim=-1)  # unit vectors
similarity = (tokens_norm @ tokens_norm.T).cpu().numpy()  # (N, N)  in [-1, 1]
distance = np.clip(1.0 - similarity, 0.0, 2.0)  # cosine distance in [0, 2]
affinity = (similarity + 1.0) / 2.0  # non-negative, for SpectralClustering

sim_offdiag = similarity.copy()
np.fill_diagonal(sim_offdiag, np.nan)
log.info(
    "Off-diagonal similarity: min=%.3f  max=%.3f  mean=%.3f",
    float(np.nanmin(sim_offdiag)),
    float(np.nanmax(sim_offdiag)),
    float(np.nanmean(sim_offdiag)),
)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

im0 = axes[0].imshow(similarity, cmap="RdYlGn", vmin=-1, vmax=1, aspect="auto")
axes[0].set_title(f"Cosine similarity  ({N_PATCHES}\u00d7{N_PATCHES} patches)")
axes[0].set_xlabel("patch index")
axes[0].set_ylabel("patch index")
plt.colorbar(im0, ax=axes[0], shrink=0.85)

im1 = axes[1].imshow(distance, cmap="viridis", vmin=0, vmax=2, aspect="auto")
axes[1].set_title(f"Cosine distance  ({N_PATCHES}\u00d7{N_PATCHES} patches)")
axes[1].set_xlabel("patch index")
plt.colorbar(im1, ax=axes[1], shrink=0.85)

plt.tight_layout()
plt.show()

In [ ]:
# ── Define clustering algorithms ───────────────────────────────────────────────
# Each tuple: (display_name, fitted_estimator, input_matrix)
tokens_l2 = sk_normalize(tokens.cpu().float().numpy())  # L2-normed features for KMeans
DBSCAN_EPS = 10.
N_CLUSTERS = 6
algo_configs: list[tuple[str, object, np.ndarray]] = [
    (
        f"DBSCAN\neps={DBSCAN_EPS}",
        DBSCAN(eps=DBSCAN_EPS, min_samples=3, metric="precomputed"),
        distance,
    ),
    (
        "Agglomerative\naverage linkage",
        AgglomerativeClustering(n_clusters=N_CLUSTERS, metric="precomputed", linkage="average"),
        distance,
    ),
    (
        "Agglomerative\ncomplete linkage",
        AgglomerativeClustering(n_clusters=N_CLUSTERS, metric="precomputed", linkage="complete"),
        distance,
    ),
    (
        "Agglomerative\nsingle linkage",
        AgglomerativeClustering(n_clusters=N_CLUSTERS, metric="precomputed", linkage="single"),
        distance,
    ),
    (
        f"KMeans  k={N_CLUSTERS}",
        KMeans(n_clusters=N_CLUSTERS, n_init=10, random_state=42),
        tokens_l2,
    ),
    (
        f"Spectral  k={N_CLUSTERS}",
        SpectralClustering(
            n_clusters=N_CLUSTERS, affinity="precomputed", random_state=42, n_init=10
        ),
        affinity,
    ),
]

# HDBSCAN — available in scikit-learn >= 1.3
try:
    from sklearn.cluster import HDBSCAN  # noqa: PLC0415

    algo_configs.append(
        (
            "HDBSCAN\nmin_cluster_size=5",
            HDBSCAN(min_cluster_size=5, metric="precomputed"),
            distance,
        )
    )
    log.info("HDBSCAN available — added to experiment")
except ImportError:
    log.warning("HDBSCAN not available (requires scikit-learn >= 1.3)")

# ── Run all algorithms ─────────────────────────────────────────────────────────
cluster_results: list[tuple[str, np.ndarray]] = []
for name, algo, X in algo_configs:
    labels = algo.fit_predict(X)
    n_found = len(set(labels) - {-1})
    n_noise = int((labels == -1).sum())
    log.info("%-44s  %d clusters  %d noise", name.replace("\n", " "), n_found, n_noise)
    cluster_results.append((name, labels))

In [ ]:
# ── Visualise cluster assignments ──────────────────────────────────────────────
OVERLAY_ALPHA = 0.60
CMAP = plt.get_cmap("tab20")  # 20 visually distinct colours

# Display image resized to the encoder's input resolution so patch boundaries align.
display_img = np.array(img.resize((IMG_SIZE, IMG_SIZE), Image.BICUBIC))


def _labels_to_rgba(labels: np.ndarray) -> np.ndarray:
    """(N,) flat labels → (H_GRID, W_GRID, 4) RGBA float32 overlay at patch resolution."""
    labels_2d = np.zeros([H_GRID, W_GRID])
    labels_2d[~fg_result['foreground_mask_feature']] = labels
    unique_clusters = sorted(c for c in set(labels_2d.flat) if c != -1)
    rgba = np.zeros((H_GRID, W_GRID, 4), dtype=np.float32)
    for i, cid in enumerate(unique_clusters):
        color = np.array(CMAP(i % 20), dtype=np.float32)
        color[3] = OVERLAY_ALPHA
        rgba[labels_2d == cid] = color
    if -1 in labels_2d:
        rgba[labels_2d == -1] = [0.05, 0.05, 0.05, OVERLAY_ALPHA]
    return rgba


def _draw_result(ax: plt.Axes, labels: np.ndarray, title: str) -> None:
    labels_2d = np.zeros([H_GRID, W_GRID])
    labels_2d[~fg_result['foreground_mask_feature']] = labels
    unique_clusters = sorted(c for c in set(labels_2d.flat) if c != -1)
    n_clusters = len(unique_clusters)
    n_noise = int((labels == -1).sum())

    # Scale patch-grid overlay to image resolution with nearest-neighbour to keep hard edges.
    overlay_small = _labels_to_rgba(labels)
    overlay_pil = Image.fromarray((overlay_small * 255).astype(np.uint8), mode="RGBA")
    overlay_arr = np.array(overlay_pil.resize((IMG_SIZE, IMG_SIZE), Image.NEAREST)) / 255.0

    ax.imshow(display_img)
    ax.imshow(overlay_arr, interpolation="nearest")

    legend_handles = [
        mpatches.Patch(color=CMAP(i % 20), label=f"cluster {cid}")
        for i, cid in enumerate(unique_clusters)
    ]
    if n_noise:
        legend_handles.append(
            mpatches.Patch(facecolor=(0.05, 0.05, 0.05), label=f"noise ({n_noise})")
        )
    ax.legend(handles=legend_handles, loc="lower right", fontsize=6, framealpha=0.85)

    suffix = f", {n_noise} noise" if n_noise else ""
    ax.set_title(f"{title}\n{n_clusters} clusters{suffix}", fontsize=9)
    ax.axis("off")


# ── Layout: header row + algorithm rows ───────────────────────────────────────
NCOLS = 3
n_algos = len(cluster_results)
n_algo_rows = (n_algos + NCOLS - 1) // NCOLS
nrows = 1 + n_algo_rows  # header row + algorithm rows

fig, axes = plt.subplots(nrows, NCOLS, figsize=(NCOLS * 5, nrows * 5))
axes = axes.flatten()

# Header row: original | similarity | distance
axes[0].imshow(display_img)
axes[0].set_title("Original image", fontsize=11)
axes[0].axis("off")

im_sim = axes[1].imshow(similarity, cmap="RdYlGn", vmin=-1, vmax=1, aspect="auto")
axes[1].set_title("Cosine similarity matrix", fontsize=11)
axes[1].set_xlabel("patch index")
axes[1].set_ylabel("patch index")
plt.colorbar(im_sim, ax=axes[1], shrink=0.80)

im_dist = axes[2].imshow(distance, cmap="viridis", vmin=0, vmax=2, aspect="auto")
axes[2].set_title("Cosine distance matrix", fontsize=11)
axes[2].set_xlabel("patch index")
plt.colorbar(im_dist, ax=axes[2], shrink=0.80)

# Algorithm subplots
for idx, (name, labels) in enumerate(cluster_results):
    _draw_result(axes[NCOLS + idx], labels, name)

# Hide unused cells
for i in range(NCOLS + n_algos, len(axes)):
    axes[i].axis("off")

plt.suptitle(
    f"DINO{DINO_VERSION}-{DINO_SIZE}  |  {N_PATCHES} patches  |  {IMG_SIZE}\u00d7{IMG_SIZE}px",
    fontsize=13,
    y=1.01,
)
plt.tight_layout()
plt.show()